In [ ]:
import os
import random
import cv2
import matplotlib.pyplot as plt

# Paths
IMG_DIR = "datasets/images/train"
LBL_DIR = "datasets/labels_bbox/train"

# Number of samples to visualize
NUM_SAMPLES = 8

# Get all image files
image_files = [f for f in os.listdir(IMG_DIR) if f.endswith(".png")]

# Randomly select samples
samples = random.sample(image_files, min(NUM_SAMPLES, len(image_files)))

def draw_yolo_bboxes(image, label_path):
    h, w, _ = image.shape

    if not os.path.exists(label_path):
        return image

    with open(label_path, "r") as f:
        lines = f.readlines()

    for line in lines:
        parts = line.strip().split()
        if len(parts) != 5:
            continue

        cls, xc, yc, bw, bh = map(float, parts)

        # Convert YOLO → pixel coords
        x1 = int((xc - bw/2) * w)
        y1 = int((yc - bh/2) * h)
        x2 = int((xc + bw/2) * w)
        y2 = int((yc + bh/2) * h)

        # Draw rectangle
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)

        # Label text
        label = f"oil"
        cv2.putText(image, label, (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                    (0, 255, 0), 1)

    return image

# Plot
cols = 4
rows = (len(samples) + cols - 1) // cols

plt.figure(figsize=(15, 8))

for i, img_name in enumerate(samples):
    img_path = os.path.join(IMG_DIR, img_name)
    lbl_path = os.path.join(LBL_DIR, img_name.replace(".png", ".txt"))

    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    image = draw_yolo_bboxes(image, lbl_path)

    plt.subplot(rows, cols, i + 1)
    plt.imshow(image)
    plt.title(img_name, fontsize=8)
    plt.axis("off")

plt.tight_layout()
plt.show()